# Домашнее задание 9. Проектирование системы целиком

Тема: ML-система для расчета складских запасов сетевого магазина.

Вариант, который я выбрал: batch + reactive retraining.

Идея простая:

1. пришел новый batch с остатками / продажами / поставками
2. Airflow дождался файла в S3/MinIO или локально
3. данные прошли validation
4. модель обучилась и сравнилась с baseline
5. если метрики норм -> регистрируем новую версию
6. если хуже -> оставляем старую

---


In [1]:
from IPython.display import display
import pandas as pd

answers = pd.DataFrame([
    {
        "кейс": "Книжный Мир / рекомендации книг",
        "архитектура": "batch retraining",
        "почему": "рекомендации можно обновлять раз в сутки, секунды тут не нужны",
    },
    {
        "кейс": "Защита Онлайн / антифрод",
        "архитектура": "streaming / online scoring",
        "почему": "транзакцию надо проверить сразу, лимит 1500 мс",
    },
    {
        "кейс": "Мой Круг / лента новостей",
        "архитектура": "hybrid: batch + быстрые online features",
        "почему": "общие интересы считаем batch-ом, последние лайки учитываем почти сразу",
    },
    {
        "кейс": "Быстрый Путь / динамические цены",
        "архитектура": "online learning / streaming pipeline",
        "почему": "спрос, машины и пробки меняются постоянно",
    },
    {
        "кейс": "Моя задача / складские запасы",
        "архитектура": "batch + reactive retraining через S3 sensor",
        "почему": "остатки обычно приходят пачками, но новый batch должен запускать DAG автоматически",
    },
])

display(answers)

print("Вывод: для складских запасов беру batch + reactive retraining.")
print("Online learning тут лишний: важнее понятный rollback, validation и сравнение с baseline.")


,кейс,архитектура,почему
0,Книжный Мир / рекомендации книг,batch retraining,"рекомендации можно обновлять раз в сутки, секу..."
1,Защита Онлайн / антифрод,streaming / online scoring,"транзакцию надо проверить сразу, лимит 1500 мс"
2,Мой Круг / лента новостей,hybrid: batch + быстрые online features,"общие интересы считаем batch-ом, последние лай..."
3,Быстрый Путь / динамические цены,online learning / streaming pipeline,"спрос, машины и пробки меняются постоянно"
4,Моя задача / складские запасы,batch + reactive retraining через S3 sensor,"остатки обычно приходят пачками, но новый batc..."


Вывод: для складских запасов беру batch + reactive retraining.
Online learning тут лишний: важнее понятный rollback, validation и сравнение с baseline.


## 2. DAG Airflow для расчета складских запасов

Для склада я выбрал Continuous Training Loop:

```text
wait_for_inventory_batch
  -> load_inventory_data
  -> validate_inventory_data
  -> train_model
  -> evaluate_model
  -> compare_with_baseline
  -> register_model / skip_deploy
  -> finish
```

Что важно:

- `wait_for_inventory_batch` ждет файл с остатками
- в локальном режиме это `FileSensor`
- в S3-режиме это `S3KeySensor` + MinIO как учебный S3
- `load_inventory_data` кладет найденный batch в `data/current_inventory_batch.csv`
- `validate_inventory_data` стопает DAG до обучения, если схема плохая
- новая модель регистрируется только если она не хуже baseline

---


In [2]:
from pathlib import Path

# Код DAG вынесен в dags/inventory_retrain_dag.py, чтобы его можно было запускать в Airflow.
# В ноутбуке печатаю фактический код, который лежит в репозитории.

dag_path = Path("dags/inventory_retrain_dag.py")
print(dag_path.read_text(encoding="utf-8"))


from __future__ import annotations

import os
import shutil
from datetime import datetime, timedelta
from pathlib import Path

try:
    from airflow import DAG
    from airflow.operators.empty import EmptyOperator
    from airflow.operators.python import BranchPythonOperator, PythonOperator
    from airflow.providers.amazon.aws.hooks.s3 import S3Hook
    from airflow.providers.amazon.aws.sensors.s3 import S3KeySensor
    from airflow.sensors.filesystem import FileSensor
    from airflow.utils.trigger_rule import TriggerRule
except ImportError:
    class DAG:
        def __init__(self, *args, **kwargs):
            pass

        def __enter__(self):
            return self

        def __exit__(self, *args):
            return False

    class _Task:
        def __init__(self, *args, **kwargs):
            self.task_id = kwargs.get("task_id", args[0] if args else self.__class__.__name__)

        def __rshift__(self, other):
            return other

    class EmptyOperator(_Task):
    

## 3. IaC и CI/CD

Минимальная инфраструктура описана через Terraform.

Я не поднимаю публичное облако ради учебного ДЗ. Поэтому использую local provider как стенд:

- manifest для входного storage (`inventory-batches`)
- manifest для MLflow-like registry/artifacts
- manifest для Airflow DAG
- manifest связки: S3 key -> рабочий batch -> validation -> branch join

CI/CD не обучает модель каждый день. Он проверяет, что код / DAG / Terraform не сломаны.

---


In [3]:
from pathlib import Path

compose_path = Path("docker-compose.yml")
print(compose_path.read_text(encoding="utf-8"))


services:
  minio:
    image: minio/minio:RELEASE.2025-04-22T22-12-26Z
    container_name: dz9_minio
    command: server /data --console-address ":9001"
    environment:
      MINIO_ROOT_USER: minioadmin
      MINIO_ROOT_PASSWORD: minioadmin
    ports:
      - "${MINIO_API_PORT:-19000}:9000"
      - "${MINIO_CONSOLE_PORT:-19001}:9001"
    volumes:
      - minio_data:/data

  minio-init:
    image: minio/mc:RELEASE.2025-04-16T18-13-26Z
    container_name: dz9_minio_init
    depends_on:
      - minio
    volumes:
      - ./data:/demo-data:ro
    entrypoint:
      - /bin/sh
      - -lc
      - |
        for i in $(seq 1 30); do
          mc alias set local http://minio:9000 minioadmin minioadmin && mc ls local >/dev/null && break
          sleep 2
        done
        mc mb --ignore-existing local/inventory-batches
        mc cp /demo-data/demo_inventory_batch.csv local/inventory-batches/incoming/2026-05-24/inventory.csv
        mc ls -r local/inventory-batches

  airflow:
    image: apac

In [4]:
from pathlib import Path

workflow_path = Path(".github/workflows/dz9-checks.yml")
print(workflow_path.read_text(encoding="utf-8"))


name: DZ9 Checks

on:
  pull_request:
  workflow_dispatch:

jobs:
  checks:
    runs-on: ubuntu-latest
    defaults:
      run:
        working-directory: .
    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"

      - name: Install Python deps
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt

      - name: Python compile
        run: python -m compileall -q src dags

      - name: Model smoke check
        run: |
          PYTHONPATH=. python - <<'PY'
          from src.inventory_data import make_demo_inventory, validate_inventory
          from src.inventory_train import train_model
          from src.inventory_evaluate import evaluate_model, compare_with_baseline

          batch_path = make_demo_inventory()
          report = validate_inventory(batch_path)
          assert report["status"] == "pass"
          train_model(batch_path)
          metrics =

In [5]:
from pathlib import Path

print("--- terraform main.tf ---")
print(Path("infra/main.tf").read_text(encoding="utf-8"))

print("\n--- terraform outputs.tf ---")
print(Path("infra/outputs.tf").read_text(encoding="utf-8"))


--- terraform main.tf ---
locals {
  base_path = "${var.artifact_root}/${var.environment}"
}

resource "local_file" "storage_manifest" {
  filename = "${local.base_path}/storage_manifest.txt"
  content  = "bucket=${var.inventory_bucket_name}\npath=incoming/YYYY-MM-DD/inventory.csv\n"
}

resource "local_file" "mlflow_manifest" {
  filename = "${local.base_path}/mlflow_manifest.txt"
  content  = "tracking_uri=file:../mlruns\nartifact_store=../models\n"
}

resource "local_file" "airflow_manifest" {
  filename = "${local.base_path}/airflow_manifest.txt"
  content  = "dag_id=inventory_retrain_pipeline\nsensor=S3KeySensor_or_local_FileSensor\n"
}

resource "local_file" "pipeline_contract_manifest" {
  filename = "${local.base_path}/pipeline_contract_manifest.txt"
  content  = <<-EOT
    s3_bucket=${var.inventory_bucket_name}
    s3_key_template=incoming/{{ ds }}/inventory.csv
    active_batch_path=data/current_inventory_batch.csv
    validation_fail_policy=stop_before_train
    branch_join_t

In [6]:
from pathlib import Path

print("--- terraform plan ---")
print(Path("reports/terraform_plan.txt").read_text(encoding="utf-8")[:2500])


--- terraform plan ---

Terraform used the selected providers to generate the following execution
plan. Resource actions are indicated with the following symbols:
  + create

Terraform will perform the following actions:

  # local_file.airflow_manifest will be created
  + resource "local_file" "airflow_manifest" {
      + content              = <<-EOT
            dag_id=inventory_retrain_pipeline
            sensor=S3KeySensor_or_local_FileSensor
        EOT
      + content_base64sha256 = (known after apply)
      + content_base64sha512 = (known after apply)
      + content_md5          = (known after apply)
      + content_sha1         = (known after apply)
      + content_sha256       = (known after apply)
      + content_sha512       = (known after apply)
      + directory_permission = "0777"
      + file_permission      = "0777"
      + filename             = "../.infra_artifacts/dev/airflow_manifest.txt"
      + id                   = (known after apply)
    }

  # local_file.mlf

In [7]:
from pathlib import Path

print("--- terraform destroy plan ---")
print(Path("reports/terraform_destroy_plan.txt").read_text(encoding="utf-8")[:1200])

print("\nВывод: деинсталляция предусмотрена через terraform destroy / destroy plan.")


--- terraform destroy plan ---

No changes. No objects need to be destroyed.

Either you have not created any objects yet or the existing objects were
already deleted outside of Terraform.


Вывод: деинсталляция предусмотрена через terraform destroy / destroy plan.


## 4. Управление рисками / SLI / SLO

Запустить модель мало. Надо еще понимать, когда система работает нормально, а когда надо вмешаться.

Я разделил SLI на 3 уровня:

1. бизнес
2. модель / данные
3. код / инфраструктура

---


| уровень | SLI | normal / SLO | warning | critical | что делаем |
|---|---|---:|---:|---:|---|
| business | доля SKU с прогнозом на завтра | `>= 95%` | `90-95%` | `< 90%` | перезапуск DAG / ручной расчет top-SKU |
| business | доля найденных дефицитных SKU | `>= 80%` за неделю | `65-80%` | `< 65%` | проверить продажи / пересмотреть модель |
| data | validation pass rate | `100%` важных проверок | warning по неважным полям | любой fail по схеме | стоп до train |
| model | MAPE прогноза остатков | `<= 15%` | `15-25%` | `> 25%` | не регистрировать модель |
| code | p95 latency прогноза | `< 300 ms` | `300-1000 ms` | `> 1000 ms` | профилировать / rollback |
| infra | successful DAG run rate | `>= 99%` daily runs | 1 падение | 2 падения подряд | смотреть Airflow / storage / registry |
| infra | S3/MinIO availability | `>= 99.5%` | retry | batch недоступен | incident по storage |

**Риски:**

- batch не пришел -> sensor timeout -> обучение не стартует
- batch плохой по схеме -> `ValueError` в validation -> train не запускаем
- новая модель хуже baseline -> `skip_deploy`
- latency выросла -> MDD test -> решение через ADR
- Terraform может удалить лишнее -> сначала смотрим `plan`, потом `apply/destroy`


## 5. Metrics Driven Development

Для MDD я взял системную метрику, не метрику модели:

```text
latency_ms для расчета прогноза запасов
```

Идея проверки:

- reference dataset = нормальная историческая latency
- new dataset = latency после изменения расчета
- H0: latency не выросла значимо
- H1: latency выросла значимо
- alpha = 0.05
- test = Mann-Whitney U, т.к. не хочу спорить про нормальность распределения

---


In [8]:
from pathlib import Path

import pandas as pd
from scipy.stats import mannwhitneyu

ref = pd.read_csv("data/reference_latency.csv")["latency_ms"]
new = pd.read_csv("data/new_latency.csv")["latency_ms"]

stat, p_value = mannwhitneyu(new, ref, alternative="greater")

print("reference rows:", len(ref))
print("new rows:", len(new))
print("reference median:", round(ref.median(), 2))
print("new median:", round(new.median(), 2))
print("stat:", round(stat, 3))
print("p-value:", round(p_value, 8))

if p_value < 0.05:
    print("Решение: latency выросла значимо, надо добавлять cache / переносить тяжелые lag features в batch preprocessing")
else:
    print("Решение: статистически значимого роста нет, пока ничего не меняю")

print("\n--- сохраненный отчет ---")
print(Path("reports/mdd_test_result.md").read_text(encoding="utf-8"))


reference rows: 48
new rows: 48
reference median: 121.0
new median: 179.68
stat: 2298.0
p-value: 0.0
Решение: latency выросла значимо, надо добавлять cache / переносить тяжелые lag features в batch preprocessing

--- сохраненный отчет ---
# MDD latency test

- metric: `latency_ms`
- reference rows: `48`
- new rows: `48`
- reference median: `121.00`
- new median: `179.69`
- test: `Mann-Whitney U`, alternative=`greater`
- alpha: `0.05`
- statistic: `2298.000`
- p_value: `0.00000000`
- decision: `add cache before stock history read`

**Вывод:** p-value ниже 0.05, рост latency считаю статистически значимым.



## ADR: рост latency прогноза запасов

**Status:** accepted

**Context:**

Есть два набора измерений latency:

- `data/reference_latency.csv` - нормальная история
- `data/new_latency.csv` - новый вариант расчета

Метрика системная: скорость расчета прогноза запасов. Это не accuracy модели, а здоровье ML-системы.

**Decision:**

- H0: latency не выросла статистически значимо
- H1: latency выросла статистически значимо
- test: Mann-Whitney U
- alpha: `0.05`
- p-value ниже `0.05`

Решение: добавить cache перед чтением истории остатков + тяжелые lag features считать заранее в batch preprocessing.

**Consequences:**

Плюсы:

- меньше p95 latency на расчете прогноза
- меньше повторных чтений истории по одному SKU
- проще держать SLO `< 300 ms`

Минусы:

- появляется cache invalidation
- надо следить за свежестью batch features

Что мониторю дальше:

- p95 latency прогноза
- доля stale cache
- MAPE после переноса lag features


## 6. Итоговый вывод

В этой работе я собрал не просто модель, а контур вокруг нее.

Для склада больше подходит batch + reactive retraining: остатки приходят пачками, но новый файл должен сам запускать DAG. Airflow отвечает за ожидание batch, validation, обучение, оценку и ветвление. S3/MinIO нужен как место, где появляется входной файл, а `data/current_inventory_batch.csv` становится рабочей копией для train. Если данные плохие, validation падает до обучения. Если модель хуже baseline, новая версия не регистрируется. Terraform описывает минимальные ресурсы и путь удаления инфраструктуры. MDD сделал по latency, т.к. в реальной ML-системе важны не только метрики модели, но и скорость / стабильность / стоимость работы.
